In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
RAW = "../data/raw/"


In [3]:
sales = pd.read_excel(RAW + "Case06_Table1_Daily_SKU_Sales_by_Store.xlsx")
repl  = pd.read_excel(RAW + "Case06_Table2_Inventory_Replenishment_Log.xlsx")
sku   = pd.read_excel(RAW + "Case06_Table3_SKU_Master.xlsx")
ctx   = pd.read_excel(RAW + "Case06_Table4_Demand_Context_and_Crosswalk.xlsx", sheet_name=None)
demand_ctx = ctx["demand_context"]
crosswalk  = ctx["pincode_city_crosswalk"]

tables = {"sales": sales, "repl": repl, "sku": sku, "demand_ctx": demand_ctx, "crosswalk": crosswalk}
for name, df in tables.items():
    print(name, df.shape, list(df.columns))


sales (438000, 5) ['date', 'store_id', 'pincode', 'sku_id', 'units_sold']
repl (438000, 8) ['date', 'store_id', 'sku_id', 'on_hand_qty_open', 'order_placed_qty', 'order_received_flag', 'qty_received', 'on_hand_qty_close']
sku (80, 7) ['sku_id', 'sku_name', 'category', 'shelf_life_value', 'shelf_life_unit', 'unit_cost', 'selling_price']
demand_ctx (2190, 5) ['date', 'city', 'rainfall_mm', 'is_local_event', 'is_public_holiday']
crosswalk (15, 2) ['pincode', 'city']


In [4]:
# Cell 3: missing values in every table
for name, df in tables.items():
    miss = df.isna().sum()
    out = pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(2)})
    print("\n====", name)
    print(out[out.missing > 0] if (out.missing > 0).any() else "no missing values")


==== sales
no missing values

==== repl
                     missing    pct
order_received_flag   223404  51.01

==== sku
no missing values

==== demand_ctx
no missing values

==== crosswalk
no missing values


In [5]:
# Cell 4: duplicate rows in every table
for name, df in tables.items():
    print(name, "duplicate rows:", df.duplicated().sum())

sales duplicate rows: 0
repl duplicate rows: 0
sku duplicate rows: 0
demand_ctx duplicate rows: 0
crosswalk duplicate rows: 0


In [6]:
# Cell 5: the keys must be unique (one row per date x store x SKU)
print("sales dup keys:", sales.duplicated(["date", "store_id", "sku_id"]).sum())
print("repl  dup keys:", repl.duplicated(["date", "store_id", "sku_id"]).sum())
print("sku   dup keys:", sku.duplicated(["sku_id"]).sum())

sales dup keys: 0
repl  dup keys: 0
sku   dup keys: 0


In [7]:
# Cell 6: convert dates and check ranges
sales["date"] = pd.to_datetime(sales["date"])
repl["date"] = pd.to_datetime(repl["date"])
demand_ctx["date"] = pd.to_datetime(demand_ctx["date"])

print("sales range:", sales.date.min(), "to", sales.date.max(), "| unique days:", sales.date.nunique())
print("repl  range:", repl.date.min(), "to", repl.date.max())
print("ctx   range:", demand_ctx.date.min(), "to", demand_ctx.date.max())

sales range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00 | unique days: 365
repl  range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00
ctx   range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00


In [8]:
# Cell 7: impossible values
print("negative units_sold:", (sales.units_sold < 0).sum())
print("negative open stock:", (repl.on_hand_qty_open < 0).sum())
print("negative close stock:", (repl.on_hand_qty_close < 0).sum())
print("negative received:", (repl.qty_received < 0).sum())
print("selling price <= unit cost:", (sku.selling_price <= sku.unit_cost).sum())
print(sales.units_sold.describe())

negative units_sold: 0
negative open stock: 0
negative close stock: 0
negative received: 0
selling price <= unit cost: 0
count    438000.000000
mean         13.251112
std          13.650451
min           0.000000
25%           5.000000
50%          10.000000
75%          18.000000
max         490.000000
Name: units_sold, dtype: float64


In [9]:
# Cell 8: count stores, SKUs, categories, cities
print("rows in SKU master:", len(sku))
print("unique SKU ids in master:", sku.sku_id.nunique())
print("unique SKU ids in sales :", sales.sku_id.nunique())
print("unique stores in sales  :", sales.store_id.nunique())
print("SKUs in sales but not in master:", set(sales.sku_id) - set(sku.sku_id))
print("SKUs in master but not in sales:", set(sku.sku_id) - set(sales.sku_id))
print("\nSKUs per category:")
print(sku.category.value_counts())
print("\ncities:", crosswalk.city.nunique(), "| pincodes:", crosswalk.pincode.nunique())

rows in SKU master: 80
unique SKU ids in master: 80
unique SKU ids in sales : 80
unique stores in sales  : 15
SKUs in sales but not in master: set()
SKUs in master but not in sales: set()

SKUs per category:
category
Tea & Coffee        6
Rice                6
Dairy - Milk        6
Bakery - Bread      6
Leafy Vegetables    6
Spices              5
Namkeen             4
Dairy - Paneer      4
Fresh Cut Fruit     4
Root Vegetables     4
Chocolates          3
Biscuits            3
Eggs                3
Non-Veg Fresh       3
Herbs               3
Cooking Oil         2
Soap & Shampoo      2
Canned Food         2
Butter & Cheese     2
Cold Beverages      2
Dairy - Curd        2
Detergent           1
Noodles & Pasta     1
Name: count, dtype: int64

cities: 6 | pincodes: 15


In [10]:
# Cell 9: shelf life comes in two units, check the mix
print(sku.shelf_life_unit.value_counts())
print(sku.groupby("shelf_life_unit").shelf_life_value.describe())

shelf_life_unit
hours    45
days     35
Name: count, dtype: int64
                 count        mean        std   min    25%    50%    75%    max
shelf_life_unit                                                                
days              35.0  201.257143  99.922129  41.0  122.5  199.0  285.5  360.0
hours             45.0   30.888889  18.853770   6.0   14.0   27.0   45.0   68.0


In [11]:
# Cell 10: is the missing order_received_flag just "no order placed"?
repl["order_placed"] = repl.order_placed_qty > 0
print(pd.crosstab(repl.order_placed, repl.order_received_flag.fillna("blank")))

order_received_flag    N       Y   blank
order_placed                            
False                  0       0  223404
True                 340  214256       0


In [12]:
# Cell 11: orders placed but never delivered
print(repl.order_received_flag.value_counts(dropna=False))

order_received_flag
NaN    223404
Y      214256
N         340
Name: count, dtype: int64


In [13]:
# Cell 12: the 490 outlier, which rows are extreme?
print(sales.nlargest(10, "units_sold"))
print("rows above 100 units:", (sales.units_sold > 100).sum())

             date store_id  pincode   sku_id  units_sold
401359 2025-08-31     DS07   560001  SKU0080         490
333359 2025-07-05     DS12   122007  SKU0080         422
394719 2025-08-25     DS14   201307  SKU0080         404
390959 2025-08-22     DS12   122007  SKU0080         403
416239 2025-09-12     DS13   201304  SKU0080         393
304239 2025-06-11     DS08   411001  SKU0080         345
382319 2025-08-15     DS09   500004  SKU0080         344
338239 2025-07-09     DS13   201304  SKU0080         339
22159  2024-10-19     DS07   560001  SKU0080         337
433919 2025-09-27     DS09   500004  SKU0080         337
rows above 100 units: 809


In [14]:
# Cell 13: are the extreme days on one SKU or on rainy days?
big = sales[sales.units_sold > 100].merge(demand_ctx.merge(crosswalk, on="city"), on=["date", "pincode"])
print(big.groupby("sku_id").size().sort_values(ascending=False).head(10))
print(big.rainfall_mm.describe())

sku_id
SKU0080    806
SKU0043      1
SKU0050      1
SKU0059      1
dtype: int64
count    809.000000
mean      37.670087
std       27.453616
min        0.000000
25%       17.800000
50%       29.200000
75%       51.300000
max      186.900000
Name: rainfall_mm, dtype: float64


In [15]:
# Cell 14: each store maps to one pincode?
print(sales.groupby("store_id").pincode.nunique().value_counts())
print(sales[["store_id", "pincode"]].drop_duplicates().merge(crosswalk, on="pincode").sort_values("city"))

pincode
1    15
Name: count, dtype: int64
   store_id  pincode       city
6      DS07   560001  Bengaluru
9      DS10   560004  Bengaluru
14     DS15   560007  Bengaluru
0      DS01   122001    Gurgaon
4      DS05   122004    Gurgaon
11     DS12   122007    Gurgaon
5      DS06   500001  Hyderabad
8      DS09   500004  Hyderabad
2      DS03   400001     Mumbai
3      DS04   400004     Mumbai
1      DS02   201301      Noida
12     DS13   201304      Noida
13     DS14   201307      Noida
7      DS08   411001       Pune
10     DS11   411004       Pune


In [16]:
# Cell 15: make the blank flag meaningful
repl["order_received_flag"] = repl.order_received_flag.fillna("NoOrder")
print(repl.order_received_flag.value_counts())

order_received_flag
NoOrder    223404
Y          214256
N             340
Name: count, dtype: int64


In [17]:
# Cell 16: dates on which onion was actually delivered, per store
ONION = "SKU0080"
onion_in = repl[(repl.sku_id == ONION) & (repl.qty_received > 0)].sort_values(["store_id", "date"]).copy()
onion_in["gap_days"] = onion_in.groupby("store_id").date.diff().dt.days
print(onion_in.gap_days.describe())
print(onion_in.gap_days.value_counts().sort_index())

count    1397.000000
mean        3.856836
std         2.014874
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        25.000000
Name: gap_days, dtype: float64
gap_days
1.0     132
2.0     194
3.0     310
4.0     341
5.0     229
6.0      83
7.0      35
8.0      33
9.0      18
10.0      9
11.0      5
12.0      4
13.0      1
14.0      1
15.0      1
25.0      1
Name: count, dtype: int64


In [18]:
# Cell 17: gap between deliveries, per store
print(onion_in.groupby("store_id").gap_days.agg(["count", "mean", "median", "min", "max"]).round(2))

          count  mean  median  min   max
store_id                                
DS01         95  3.80     4.0  1.0  11.0
DS02         91  3.95     4.0  1.0  14.0
DS03         94  3.81     4.0  1.0   9.0
DS04         93  3.85     4.0  1.0   9.0
DS05         92  3.96     4.0  1.0  11.0
DS06         94  3.80     4.0  1.0   9.0
DS07         94  3.81     4.0  1.0  12.0
DS08         95  3.78     3.0  1.0  12.0
DS09         95  3.77     4.0  1.0  10.0
DS10         93  3.86     4.0  1.0  12.0
DS11         94  3.83     4.0  1.0  12.0
DS12         95  3.83     4.0  1.0  10.0
DS13         91  3.93     4.0  1.0  15.0
DS14         91  3.92     4.0  1.0  13.0
DS15         90  3.98     4.0  1.0  25.0


In [19]:
# Cell 18: the lag between placing an order and receiving it (order-to-arrival time)
o = repl[repl.sku_id == ONION].sort_values(["store_id", "date"]).copy()
o["placed_date"] = o.date.where(o.order_placed_qty > 0)
o["received_date"] = o.date.where(o.qty_received > 0)

# for each delivery, look back at the most recent order placed on or before that day
o["last_order_date"] = o.groupby("store_id").placed_date.ffill()
arrivals = o[o.qty_received > 0].copy()
arrivals["lead_days"] = (arrivals.date - arrivals.last_order_date).dt.days
print(arrivals.lead_days.describe())
print(arrivals.lead_days.value_counts().sort_index())

count    1412.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: lead_days, dtype: float64
lead_days
0    1412
Name: count, dtype: int64


In [20]:
# Cell 19
ONION_LIFE_DAYS = int(round(onion_in.gap_days.mean()))   # mean = 3.86 -> 4
print("Onion shelf life =", ONION_LIFE_DAYS, "days")

Onion shelf life = 4 days


In [21]:
# Cell 20: merge all tables into one panel (one row per date x store x SKU)
panel = sales.merge(repl, on=["date", "store_id", "sku_id"], validate="1:1")
panel = panel.merge(crosswalk, on="pincode", validate="m:1")
panel = panel.merge(demand_ctx, on=["date", "city"], validate="m:1")
panel = panel.merge(sku, on="sku_id", validate="m:1")
print(panel.shape)   # expect 438000 rows

(438000, 21)


In [22]:
# Cell 21: shelf life in days for every SKU, with the onion rule applied
panel["shelf_life_days"] = np.where(panel.shelf_life_unit == "hours",
                                    panel.shelf_life_value / 24,
                                    panel.shelf_life_value)
panel.loc[panel.sku_id == ONION, "shelf_life_days"] = ONION_LIFE_DAYS

print(panel.groupby("sku_id").shelf_life_days.first().describe())
print(panel[panel.sku_id == ONION].shelf_life_days.unique())

count     80.000000
mean      88.813021
std      119.398424
min        0.250000
25%        1.031250
50%        2.604167
75%      184.250000
max      360.000000
Name: shelf_life_days, dtype: float64
[4.]


In [23]:
# Cell 22: derived columns used in every later step
panel["revenue"] = panel.units_sold * panel.selling_price
panel["margin"] = panel.selling_price - panel.unit_cost
panel["empty_at_close"] = panel.on_hand_qty_close <= 0
panel["empty_at_open"] = panel.on_hand_qty_open <= 0
panel["rainy"] = panel.rainfall_mm > 0
panel["dow"] = panel.date.dt.dayofweek
panel["month"] = panel.date.dt.to_period("M")

C:\Users\ssang\AppData\Local\Temp\ipykernel_7812\279955676.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  panel["dow"] = panel.date.dt.dayofweek


In [24]:
# Cell 23: stock-ledger check (opening - sold = closing); expect about 0 broken rows
resid = panel.on_hand_qty_open - panel.units_sold - panel.on_hand_qty_close
print("rows where the ledger breaks:", (resid.abs() > 0.5).sum())

rows where the ledger breaks: 0


In [25]:
panel["dow"] = panel.date.dt.day_of_week

In [26]:
# Cell 24: which SKU does each store SELL the most?
store_sku_sales = panel.groupby(["store_id", "sku_id"]).units_sold.sum().reset_index()
top_sold = store_sku_sales.loc[store_sku_sales.groupby("store_id").units_sold.idxmax()]
print(top_sold.sort_values("store_id"))

     store_id   sku_id  units_sold
79       DS01  SKU0080       15636
159      DS02  SKU0080       14951
239      DS03  SKU0080       16976
319      DS04  SKU0080       17308
399      DS05  SKU0080       16694
479      DS06  SKU0080       13974
559      DS07  SKU0080       21752
639      DS08  SKU0080       16100
719      DS09  SKU0080       13131
799      DS10  SKU0080       21027
879      DS11  SKU0080       17140
959      DS12  SKU0080       16142
1039     DS13  SKU0080       15886
1119     DS14  SKU0080       15317
1199     DS15  SKU0080       20606


In [27]:
# Cell 25: which SKU does each store HOLD the most (average opening stock)?
store_sku_stock = panel.groupby(["store_id", "sku_id"]).on_hand_qty_open.mean().reset_index()
top_stock = store_sku_stock.loc[store_sku_stock.groupby("store_id").on_hand_qty_open.idxmax()]
print(top_stock.sort_values("store_id"))

     store_id   sku_id  on_hand_qty_open
79       DS01  SKU0080        120.791781
159      DS02  SKU0080        121.205479
239      DS03  SKU0080        130.356164
319      DS04  SKU0080        133.630137
399      DS05  SKU0080        142.408219
479      DS06  SKU0080        103.002740
559      DS07  SKU0080        178.547945
639      DS08  SKU0080        120.958904
719      DS09  SKU0080         94.460274
799      DS10  SKU0080        163.104110
879      DS11  SKU0080        133.849315
959      DS12  SKU0080        118.663014
1039     DS13  SKU0080        128.093151
1119     DS14  SKU0080        127.893151
1199     DS15  SKU0080        178.263014


In [28]:
# Cell 26: is the SKU a store holds most also the one it sells most?
cmp = top_sold.merge(top_stock, on="store_id", suffixes=("_sold", "_held"))
cmp["same_sku"] = cmp.sku_id_sold == cmp.sku_id_held
print(cmp[["store_id", "sku_id_sold", "sku_id_held", "same_sku"]])
print("stores where they match:", cmp.same_sku.sum(), "of", len(cmp))

   store_id sku_id_sold sku_id_held  same_sku
0      DS01     SKU0080     SKU0080      True
1      DS02     SKU0080     SKU0080      True
2      DS03     SKU0080     SKU0080      True
3      DS04     SKU0080     SKU0080      True
4      DS05     SKU0080     SKU0080      True
5      DS06     SKU0080     SKU0080      True
6      DS07     SKU0080     SKU0080      True
7      DS08     SKU0080     SKU0080      True
8      DS09     SKU0080     SKU0080      True
9      DS10     SKU0080     SKU0080      True
10     DS11     SKU0080     SKU0080      True
11     DS12     SKU0080     SKU0080      True
12     DS13     SKU0080     SKU0080      True
13     DS14     SKU0080     SKU0080      True
14     DS15     SKU0080     SKU0080      True
stores where they match: 15 of 15


In [ ]:
# Cell 27: revenue by category, city and store
print(panel.groupby("category").revenue.sum().sort_values(ascending=False).round(0))

category
Bakery - Bread      137824150.0
Dairy - Milk        137389033.0
Rice                 80783815.0
Dairy - Paneer       60936675.0
Fresh Cut Fruit      53997439.0
Dairy - Curd         53833980.0
Eggs                 49327956.0
Tea & Coffee         40481332.0
Cold Beverages       36226475.0
Leafy Vegetables     35296940.0
Soap & Shampoo       31119053.0
Herbs                27694309.0
Non-Veg Fresh        26288439.0
Canned Food          26218373.0
Root Vegetables      25075579.0
Chocolates           22679867.0
Detergent            18659586.0
Namkeen              16044337.0
Spices               13129952.0
Biscuits             10861604.0
Noodles & Pasta       8545535.0
Cooking Oil           8088521.0
Butter & Cheese       5813279.0
Name: revenue, dtype: float64
city
Bengaluru    185411880.0
Gurgaon      185076834.0
Noida        184215860.0
Mumbai       124010125.0
Pune         123853795.0
Hyderabad    123747735.0
Name: revenue, dtype: float64
store_id
DS12    62566192.0
DS06    6239

In [30]:
print(panel.groupby("city").revenue.sum().sort_values(ascending=False).round(0))

city
Bengaluru    185411880.0
Gurgaon      185076834.0
Noida        184215860.0
Mumbai       124010125.0
Pune         123853795.0
Hyderabad    123747735.0
Name: revenue, dtype: float64


In [31]:
print(panel.groupby("store_id").revenue.sum().sort_values(ascending=False).round(0))

store_id
DS12    62566192.0
DS06    62395343.0
DS07    62223301.0
DS04    62067068.0
DS08    61985666.0
DS03    61943057.0
DS11    61868129.0
DS15    61818074.0
DS13    61628163.0
DS10    61370504.0
DS14    61365949.0
DS05    61357207.0
DS09    61352393.0
DS02    61221747.0
DS01    61153435.0
Name: revenue, dtype: float64


In [32]:
# Cell 28: top 10 and bottom 10 SKUs by revenue, with names
sku_rev = panel.groupby(["sku_id", "sku_name"]).revenue.sum().sort_values(ascending=False)
print(sku_rev.head(10))
print(sku_rev.tail(10))

sku_id   sku_name                 
SKU0055  Bakery - Bread - Item 055    48249127.98
SKU0050  Dairy - Milk - Item 050      43060015.60
SKU0041  Dairy - Curd - Item 041      40727122.45
SKU0059  Bakery - Bread - Item 059    37393162.86
SKU0053  Dairy - Milk - Item 053      36631647.36
SKU0060  Eggs - Item 060              30884560.20
SKU0005  Soap & Shampoo - Item 005    30388369.20
SKU0057  Dairy - Milk - Item 057      26772165.08
SKU0043  Dairy - Paneer - Item 043    26644304.06
SKU0056  Dairy - Paneer - Item 056    24319568.28
Name: revenue, dtype: float64
sku_id   sku_name                   
SKU0062  Non-Veg Fresh - Item 062       1831605.16
SKU0003  Spices - Item 003              1687852.31
SKU0014  Cooking Oil - Item 014         1540210.20
SKU0025  Spices - Item 025              1467748.62
SKU0078  Leafy Vegetables - Item 078    1358988.05
SKU0047  Dairy - Paneer - Item 047      1285348.77
SKU0066  Herbs - Item 066               1144864.08
SKU0017  Spices - Item 017              1

In [33]:
# Cell 29: empty-shelf rate at close, by category and store
print(panel.groupby("category").empty_at_close.mean().round(3).sort_values(ascending=False))
print(panel.groupby("store_id").empty_at_close.mean().round(3).sort_values(ascending=False))
print("overall at close:", panel.empty_at_close.mean().round(3))
print("overall at open :", panel.empty_at_open.mean().round(3))

category
Herbs               0.719
Leafy Vegetables    0.712
Fresh Cut Fruit     0.668
Non-Veg Fresh       0.667
Root Vegetables     0.557
Butter & Cheese     0.502
Biscuits            0.487
Noodles & Pasta     0.460
Canned Food         0.455
Spices              0.446
Tea & Coffee        0.422
Cold Beverages      0.416
Cooking Oil         0.416
Namkeen             0.409
Bakery - Bread      0.402
Dairy - Paneer      0.393
Eggs                0.374
Dairy - Milk        0.354
Soap & Shampoo      0.345
Dairy - Curd        0.338
Rice                0.334
Detergent           0.312
Chocolates          0.304
Name: empty_at_close, dtype: float64
store_id
DS01    0.474
DS13    0.471
DS08    0.468
DS02    0.467
DS10    0.467
DS05    0.466
DS03    0.466
DS14    0.465
DS11    0.464
DS04    0.461
DS12    0.461
DS15    0.455
DS06    0.455
DS07    0.453
DS09    0.453
Name: empty_at_close, dtype: float64
overall at close: 0.463
overall at open : 0.093


In [34]:
# Cell 30: demand drivers: rain, weekday, month, event, holiday
print(panel.groupby("rainy").units_sold.mean().round(2))
print(panel.groupby("dow").units_sold.mean().round(2))
print(panel.groupby("month").units_sold.mean().round(2))
print(panel.groupby("is_local_event").units_sold.mean().round(2))
print(panel.groupby("is_public_holiday").units_sold.mean().round(2))

rainy
False    12.90
True     14.52
Name: units_sold, dtype: float64
dow
0    12.14
1    12.85
2    13.07
3    12.83
4    12.91
5    14.92
6    14.05
Name: units_sold, dtype: float64
month
2024-10    13.91
2024-11    13.00
2024-12    12.81
2025-01    12.85
2025-02    12.84
2025-03    12.81
2025-04    12.79
2025-05    12.95
2025-06    13.66
2025-07    13.73
2025-08    13.96
2025-09    13.67
Freq: M, Name: units_sold, dtype: float64
is_local_event
0    13.25
1    13.54
Name: units_sold, dtype: float64
is_public_holiday
0    13.25
1    12.95
Name: units_sold, dtype: float64


In [35]:
# Cell 31: the onion under rain: sales vs empty shelves
onion = panel[panel.sku_id == ONION]
print(onion.groupby("rainy").units_sold.mean().round(2))
print(onion.groupby("rainy").empty_at_close.mean().round(3))

rainy
False     19.34
True     142.23
Name: units_sold, dtype: float64
rainy
False    0.086
True     0.536
Name: empty_at_close, dtype: float64
